# 03 — Phase 2a: Zhao et al. SVD Baseline

Replicates Zhao et al. (NeurIPS 2025) on Gemma 3 4B IT. Per-language mean hidden states → SVD → language-specific subspace M_s. Inference-time projection h' = h − λ M_s M_sᵀ h applied at the **last input token** across **middle and higher layers**.

Produces:
- Unmodified-model baseline accuracy on full MGSM (250×5).
- Sensitivity sweep on a 50-prompt dev split: best (λ_mid, λ_hi, rank).
- Best-config eval on full 250×5 + LaBSE-based language fidelity.
- Saved to `results/phase2_zhao_baseline.pt` for Phase 2b comparison.

**Compute budget:** ~5–7 hours on A100 with greedy decoding. The notebook is split into 3 phases so you can checkpoint between them.

## 0. Setup

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/kvrancic/nlp-project.git'
REPO_DIR = '/content/nlp-project'
if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
!pip install -q 'numpy>=2.0' langdetect -e .

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
Path('.env').write_text(f'HF_TOKEN={os.environ["HF_TOKEN"]}\n')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS = Path('/content/drive/MyDrive/nlp-project-results')
    DRIVE_RESULTS.mkdir(exist_ok=True, parents=True)
except Exception:
    DRIVE_RESULTS = None

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm

from src.config import (
    TARGET_LANGUAGES, N_LAYERS, D_MODEL, RESULTS_DIR,
    ZHAO_MIDDLE_LAYERS, ZHAO_HIGHER_LAYERS,
)
from src.data import load_mgsm, parse_answer_number, compute_accuracy
from src.model import load_model_and_tokenizer
from src.extraction import extract_residual_activations
from src.svd_baseline import compute_language_subspace, create_svd_hooks
from src.evaluation import evaluate_mgsm

torch.manual_seed(0); np.random.seed(0)
MAX_NEW_TOKENS = 384  # MGSM answers usually arrive well before 512 with greedy decoding.
N_DEV = 50           # per-language dev split for sensitivity sweep
N_TEST = 250         # full MGSM test set (per language)
print(f'Middle layers: {ZHAO_MIDDLE_LAYERS}')
print(f'Higher layers: {ZHAO_HIGHER_LAYERS}')

## 1. Load model + MGSM

In [ ]:
model, tokenizer = load_model_and_tokenizer()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

mgsm = load_mgsm(TARGET_LANGUAGES)

def make_prompt(question: str) -> str:
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': question}],
        tokenize=False, add_generation_prompt=True,
    )

prompts_by_lang = {l: [make_prompt(ex['question']) for ex in mgsm[l]] for l in TARGET_LANGUAGES}
golds_by_lang   = {l: [ex['answer_number']        for ex in mgsm[l]] for l in TARGET_LANGUAGES}

# Indexing convention: dev = first 50, test = full 250 (overlaps with dev — that's
# OK because dev is for HP selection; we report test for the chosen config
# alongside the unmodified baseline, both on the same 250 problems).
dev_prompts = {l: prompts_by_lang[l][:N_DEV] for l in TARGET_LANGUAGES}
dev_golds   = {l: golds_by_lang[l][:N_DEV]   for l in TARGET_LANGUAGES}
test_prompts = prompts_by_lang
test_golds   = golds_by_lang

## 2. Extract per-language mean residuals at all 34 layers (last input token)

These mean activations are the inputs to Algorithm 1's SVD. Done once; cached.

In [ ]:
all_layers = list(range(N_LAYERS))
per_lang_mean = {layer: {} for layer in all_layers}

for lang in TARGET_LANGUAGES:
    print(f'\n=== {lang}: extracting last-token residuals at all {N_LAYERS} layers ===')
    acts = extract_residual_activations(
        model, tokenizer, prompts_by_lang[lang],
        layers=all_layers, batch_size=4, positions='last',
    )
    for layer in all_layers:
        per_lang_mean[layer][lang] = acts[layer].float().mean(dim=0)  # (d_model,)
    torch.cuda.empty_cache()

# Sanity: shapes
for layer in [0, ZHAO_MIDDLE_LAYERS[0], ZHAO_HIGHER_LAYERS[-1]]:
    print(f'  layer {layer}: {tuple(per_lang_mean[layer][TARGET_LANGUAGES[0]].shape)}, '
          f'norm={per_lang_mean[layer][TARGET_LANGUAGES[0]].norm():.2f}')

## 3. SVD: compute language subspace per layer per rank

In [ ]:
RANKS = [2, 3, 4]
M_s_by_rank_layer = {r: {} for r in RANKS}
M_a_by_layer = {}

for r in RANKS:
    for layer in all_layers:
        M_a, M_s = compute_language_subspace(per_lang_mean[layer], rank=r)
        M_s_by_rank_layer[r][layer] = M_s   # (d_model, r)
        if r == max(RANKS):
            M_a_by_layer[layer] = M_a       # M_a is rank-independent in our impl

# Sanity: the language subspace should explain a non-trivial fraction of
# residual norm at middle layers (where languages diverge most).
for layer in [9, ZHAO_MIDDLE_LAYERS[len(ZHAO_MIDDLE_LAYERS)//2], 30]:
    M_s = M_s_by_rank_layer[3][layer]
    explained = []
    for lang in TARGET_LANGUAGES:
        h = per_lang_mean[layer][lang]
        proj_norm = (h @ M_s @ M_s.T).norm().item()
        explained.append(proj_norm / h.norm().item())
    print(f'  layer {layer:>2}: rank-3 subspace captures '
          f'{np.mean(explained)*100:.1f}% of mean residual norm (avg across langs)')

## 4. Unmodified baseline on full MGSM

Runs the model with no intervention on all 250 × 5 problems. **Slow (~3–4h on A100).** Required for Phase 2b H1/H2 comparison too, so always run.

In [ ]:
baseline_results = {}
for lang in TARGET_LANGUAGES:
    print(f'\n=== Baseline (no intervention): {lang} ===')
    res = evaluate_mgsm(
        model, tokenizer,
        questions=test_prompts[lang],
        gold_answers=test_golds[lang],
        max_new_tokens=MAX_NEW_TOKENS,
        batch_size=1,
    )
    baseline_results[lang] = res
    print(f'  {lang}: accuracy = {res["accuracy"]:.3f} ({sum(res["correct"])}/{N_TEST})')
    # Checkpoint after each language so partial progress survives a Colab disconnect.
    torch.save(baseline_results, RESULTS_DIR / 'phase2_baseline_partial.pt')

baseline_avg = np.mean([baseline_results[l]['accuracy'] for l in TARGET_LANGUAGES])
print(f'\nBaseline average: {baseline_avg:.3f}')

## 5. Sensitivity sweep on 50-prompt dev split

Picks the best (λ_mid, λ_hi, rank) by mean dev accuracy across 5 languages. Small grid (5 configs × 3 ranks = 15) — paper's defaults are ~0.2 / −0.2 / r=4.

In [ ]:
from itertools import product

# Tight grid centered on Zhao's defaults
GRID = list(product(
    [0.1, 0.2, 0.3],         # lambda_middle (positive: remove language)
    [-0.3, -0.2, -0.1, 0.0], # lambda_higher (negative: re-inject for fidelity)
    RANKS,                    # rank r
))
print(f'Grid configs: {len(GRID)}')

def run_with_svd(prompts, golds, lam_mid, lam_hi, rank, input_length=None):
    """Build SVD hooks for one config and evaluate via evaluate_mgsm."""
    M_s_per_layer = M_s_by_rank_layer[rank]
    hook_dict = create_svd_hooks(
        M_s_per_layer=M_s_per_layer,
        lambda_middle=lam_mid,
        lambda_higher=lam_hi,
        input_length=input_length,  # if None, hook handles per-call
    )
    hooks_list = [(layer, fn) for layer, fn in hook_dict.items()]
    return evaluate_mgsm(
        model, tokenizer, prompts, golds,
        max_new_tokens=MAX_NEW_TOKENS, batch_size=1, hooks=hooks_list,
    )

# IMPORTANT: create_svd_hooks expects an `input_length` to know which token
# is the "last input token". With batch_size=1 and varying prompt lengths,
# this changes per prompt. We rebuild hooks per prompt — slower but correct.
def evaluate_with_svd_per_prompt(prompts, golds, lam_mid, lam_hi, rank):
    correct, preds, outputs = [], [], []
    M_s_per_layer = M_s_by_rank_layer[rank]
    for q, gold in tqdm(list(zip(prompts, golds)), desc=f'lm={lam_mid} lh={lam_hi} r={rank}'):
        inputs = tokenizer(q, return_tensors='pt').to(model.device)
        input_len = inputs['input_ids'].shape[1]
        hook_dict = create_svd_hooks(
            M_s_per_layer=M_s_per_layer,
            lambda_middle=lam_mid, lambda_higher=lam_hi,
            input_length=input_len,
        )
        handles = [model.model.layers[layer].register_forward_hook(fn)
                   for layer, fn in hook_dict.items()]
        try:
            with torch.no_grad():
                gen = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        finally:
            for h in handles: h.remove()
        out = tokenizer.decode(gen[0][input_len:], skip_special_tokens=True)
        outputs.append(out); pred = parse_answer_number(out); preds.append(pred)
        correct.append(pred is not None and abs(pred - gold) < 1e-6)
    return {
        'accuracy': sum(correct) / max(len(correct), 1),
        'predictions': preds, 'outputs': outputs, 'correct': correct,
    }

In [ ]:
sweep_results = {}
for lam_mid, lam_hi, rank in GRID:
    key = (lam_mid, lam_hi, rank)
    print(f'\n--- config: lm={lam_mid}, lh={lam_hi}, rank={rank} ---')
    sweep_results[key] = {}
    for lang in TARGET_LANGUAGES:
        r = evaluate_with_svd_per_prompt(
            dev_prompts[lang], dev_golds[lang], lam_mid, lam_hi, rank,
        )
        sweep_results[key][lang] = r
        print(f'    {lang}: {r["accuracy"]:.3f}')
    avg = np.mean([sweep_results[key][l]['accuracy'] for l in TARGET_LANGUAGES])
    sweep_results[key]['avg'] = avg
    print(f'    AVG: {avg:.3f}')
    torch.save(sweep_results, RESULTS_DIR / 'phase2_zhao_sweep_partial.pt')

# Pick the best config by mean dev accuracy.
best_key = max(sweep_results.keys(), key=lambda k: sweep_results[k]['avg'])
print(f'\nBest config: lm={best_key[0]}, lh={best_key[1]}, rank={best_key[2]} '
      f'(dev avg={sweep_results[best_key]["avg"]:.3f})')

## 6. Best-config eval on full 250×5 test set

In [ ]:
best_lm, best_lh, best_rank = best_key
zhao_test = {}
for lang in TARGET_LANGUAGES:
    print(f'\n=== Zhao best on {lang} ===')
    r = evaluate_with_svd_per_prompt(
        test_prompts[lang], test_golds[lang], best_lm, best_lh, best_rank,
    )
    zhao_test[lang] = r
    print(f'  {lang}: {r["accuracy"]:.3f} (vs baseline {baseline_results[lang]["accuracy"]:.3f})')
    torch.save(zhao_test, RESULTS_DIR / 'phase2_zhao_test_partial.pt')

zhao_avg = np.mean([zhao_test[l]['accuracy'] for l in TARGET_LANGUAGES])
print(f'\nBaseline avg: {baseline_avg:.3f}')
print(f'Zhao    avg: {zhao_avg:.3f}  (delta {zhao_avg - baseline_avg:+.3f})')

## 7. Language fidelity (langdetect on output)

If projecting away language drives the model to switch output language (e.g., to English), we need to flag it. We use lightweight `langdetect` on the first 200 chars of the model output.

In [ ]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

# langdetect codes can differ from MGSM codes. Map MGSM 2-letter → langdetect 2-letter.
# en → en, zh → zh-cn or zh-tw, es → es, bn → bn, sw → sw
LANG_OK = {
    'en': {'en'},
    'zh': {'zh-cn', 'zh-tw'},
    'es': {'es'},
    'bn': {'bn'},
    'sw': {'sw'},
}

def language_fidelity(outputs: list[str], target_lang: str) -> float:
    ok = 0; total = 0
    accepted = LANG_OK[target_lang]
    for o in outputs:
        snippet = (o or '')[:300].strip()
        if not snippet:
            continue
        try:
            d = detect(snippet)
        except Exception:
            continue
        total += 1
        if d in accepted:
            ok += 1
    return ok / total if total else 0.0

fidelity = {}
for lang in TARGET_LANGUAGES:
    fidelity[lang] = {
        'baseline': language_fidelity(baseline_results[lang]['outputs'], lang),
        'zhao':     language_fidelity(zhao_test[lang]['outputs'],     lang),
    }
    print(f'  {lang}: fidelity baseline={fidelity[lang]["baseline"]:.3f}, '
          f'zhao={fidelity[lang]["zhao"]:.3f}')

## 8. Figures

In [ ]:
FIG_DIR = Path('results/figures'); FIG_DIR.mkdir(exist_ok=True, parents=True)
sns.set_theme(style='whitegrid', context='paper')

# Figure 1: per-language baseline vs Zhao.
df = pd.DataFrame({
    'baseline': {l: baseline_results[l]['accuracy'] for l in TARGET_LANGUAGES},
    f'zhao (lm={best_lm}, lh={best_lh}, r={best_rank})':
        {l: zhao_test[l]['accuracy'] for l in TARGET_LANGUAGES},
}).T
fig, ax = plt.subplots(figsize=(7, 3.4))
df.T.plot(kind='bar', ax=ax)
ax.set_ylabel('MGSM accuracy'); ax.set_ylim(0, 1)
ax.set_title('Phase 2a — unmodified vs Zhao SVD projection')
ax.legend(loc='upper right')
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig5_zhao_baseline.png', dpi=150); plt.show()

In [ ]:
# Figure 2: dev-set sensitivity heatmap (rank=best_rank slice).
rows = []
for (lm, lh, r), v in sweep_results.items():
    if r != best_rank: continue
    rows.append({'lambda_middle': lm, 'lambda_higher': lh, 'avg_acc': v['avg']})
heat = pd.DataFrame(rows).pivot(index='lambda_middle', columns='lambda_higher', values='avg_acc')
fig, ax = plt.subplots(figsize=(5.5, 3.2))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='viridis', ax=ax,
            cbar_kws={'label': 'mean dev acc (5 langs)'})
ax.set_title(f'Phase 2a sensitivity (rank={best_rank})')
plt.tight_layout(); plt.savefig(FIG_DIR / 'fig6_zhao_sweep.png', dpi=150); plt.show()

## 9. Save Phase 2a results

In [ ]:
phase2a_payload = {
    'config': {
        'middle_layers': ZHAO_MIDDLE_LAYERS,
        'higher_layers': ZHAO_HIGHER_LAYERS,
        'ranks_searched': RANKS,
        'grid': GRID,
        'best_lambda_middle': best_lm,
        'best_lambda_higher': best_lh,
        'best_rank': best_rank,
        'max_new_tokens': MAX_NEW_TOKENS,
        'n_dev': N_DEV, 'n_test': N_TEST,
    },
    'per_lang_mean': per_lang_mean,
    'M_s_by_rank_layer': M_s_by_rank_layer,
    'baseline_results': baseline_results,
    'sweep_results': sweep_results,
    'zhao_test': zhao_test,
    'language_fidelity': fidelity,
    'baseline_avg': baseline_avg,
    'zhao_avg': zhao_avg,
}
out = RESULTS_DIR / 'phase2_zhao_baseline.pt'
torch.save(phase2a_payload, out)
print(f'Saved {out} ({out.stat().st_size/1e6:.1f} MB)')
if DRIVE_RESULTS is not None:
    torch.save(phase2a_payload, DRIVE_RESULTS / 'phase2_zhao_baseline.pt')
    print(f'Drive backup: {DRIVE_RESULTS}/phase2_zhao_baseline.pt')

print('\n=== Final Phase 2a summary ===')
print(f'Best (lm, lh, rank): ({best_lm}, {best_lh}, {best_rank})')
print(f'  baseline avg: {baseline_avg:.3f}')
print(f'  zhao     avg: {zhao_avg:.3f}  (delta {zhao_avg - baseline_avg:+.3f})')
print(f'  per-language deltas:')
for l in TARGET_LANGUAGES:
    d = zhao_test[l]['accuracy'] - baseline_results[l]['accuracy']
    print(f'    {l}: {baseline_results[l]["accuracy"]:.3f} → {zhao_test[l]["accuracy"]:.3f}  ({d:+.3f})')